# Agent Handoff Chains

A handoff chain is a linear pipeline where each agent enriches a shared context object and passes it forward to the next. Unlike a `Crew`, the handoff is explicit — each agent receives a structured `Handoff` object containing the original input plus everything previous agents produced.

**What you'll build:** A three-stage pipeline where a researcher gathers facts, a writer drafts an article, and an editor polishes it — each stage building directly on the previous one's output.

**Time:** ~20 min | **Difficulty:** Intermediate

## Prerequisites & Setup

You need an `OPENAI_API_KEY` environment variable set.

**What you'll learn:**
- Create a `HandoffChain` with ordered agents
- Understand the `Handoff` context object each agent receives
- Add custom fields to the handoff for structured inter-agent communication
- Inspect intermediate results at each stage

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key before running
# os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 1: Define the Shared Handoff Context

The context dataclass accumulates results across all stages. Each agent reads the fields set by its predecessors and adds its own.

In [ ]:
from __future__ import annotations
import asyncio
from dataclasses import dataclass, field

from synapsekit.agents import Agent, HandoffChain, Handoff
from synapsekit.llms.openai import OpenAILLM
from synapsekit import LLMConfig

# The context dataclass accumulates results across all stages.
# Each agent reads the fields set by its predecessors and adds its own.
@dataclass
class ArticleContext:
    topic: str
    target_word_count: int = 500

    # Populated by the researcher
    research_notes: str = ""
    key_claims: list[str] = field(default_factory=list)

    # Populated by the writer
    draft: str = ""
    word_count: int = 0

    # Populated by the editor
    final_article: str = ""
    edits_summary: str = ""

## Step 2: Define Agents with Handoff-Aware Instructions

Each agent has instructions that describe its role in the pipeline.

In [ ]:
llm = OpenAILLM(model="gpt-4o-mini", config=LLMConfig(temperature=0.6))

researcher = Agent(
    name="researcher",
    instructions=(
        "You are a research analyst. You will receive a topic. "
        "Respond with 5 concise bullet points covering the most important facts, "
        "statistics, or insights. Keep each bullet under 25 words."
    ),
    llm=llm,
)

writer = Agent(
    name="writer",
    instructions=(
        "You are a content writer. You will receive research notes. "
        "Write an article of approximately {target_word_count} words. "
        "Use an engaging introduction, three substantive body paragraphs, and a conclusion. "
        "Do not pad — every sentence should add value."
    ),
    llm=llm,
)

editor = Agent(
    name="editor",
    instructions=(
        "You are a senior editor. You will receive a draft article. "
        "Improve clarity, fix awkward phrasing, tighten verbose sentences, "
        "and ensure consistent tone. Return the polished article followed by "
        "a one-sentence summary of the changes you made."
    ),
    llm=llm,
)

## Step 3: Define Node Functions for Each Stage

Each node function receives a `Handoff[ArticleContext]` and returns the updated context.

In [ ]:
async def research_node(handoff: Handoff[ArticleContext]) -> ArticleContext:
    ctx = handoff.context
    response = await researcher.arun(f"Topic: {ctx.topic}")

    # Store the raw text and parse out individual bullets
    ctx.research_notes = response.text
    ctx.key_claims = [
        line.strip("\u2022- ").strip()
        for line in response.text.splitlines()
        if line.strip().startswith(("\u2022", "-", "*"))
    ]
    print(f"[researcher] Produced {len(ctx.key_claims)} key claims.")
    return ctx


async def write_node(handoff: Handoff[ArticleContext]) -> ArticleContext:
    ctx = handoff.context
    prompt = (
        f"Write a {ctx.target_word_count}-word article on '{ctx.topic}'.\n\n"
        f"Use these research notes:\n{ctx.research_notes}"
    )
    response = await writer.arun(prompt)

    ctx.draft = response.text
    ctx.word_count = len(response.text.split())
    print(f"[writer] Draft is {ctx.word_count} words.")
    return ctx


async def edit_node(handoff: Handoff[ArticleContext]) -> ArticleContext:
    ctx = handoff.context
    prompt = f"Edit the following article:\n\n{ctx.draft}"
    response = await editor.arun(prompt)

    # The editor returns the polished article + a one-sentence summary
    parts = response.text.rsplit("\n\n", 1)
    ctx.final_article = parts[0]
    ctx.edits_summary = parts[1] if len(parts) > 1 else ""
    print(f"[editor] Edits summary: {ctx.edits_summary}")
    return ctx

## Step 4: Build and Run the Chain

Nodes execute left-to-right; each receives the context returned by its predecessor.

In [ ]:
async def run_pipeline(topic: str) -> ArticleContext:
    initial_context = ArticleContext(topic=topic, target_word_count=500)

    chain = HandoffChain(
        name="article-pipeline",
        # Nodes execute left-to-right; each receives the context returned by its predecessor
        nodes=[research_node, write_node, edit_node],
    )

    final_context = await chain.arun(initial_context)
    return final_context

## Complete Working Example

Run the full handoff pipeline and inspect the key claims, final article, and editor statistics.

In [ ]:
async def main():
    topic = "How renewable energy is reshaping global electricity markets"
    result = await run_pipeline(topic)

    print("\n--- KEY CLAIMS ---")
    for i, claim in enumerate(result.key_claims, 1):
        print(f"  {i}. {claim}")

    print("\n--- FINAL ARTICLE ---")
    print(result.final_article)

    print(f"\n--- STATS ---")
    print(f"  Draft word count:  {result.word_count}")
    print(f"  Editor notes:      {result.edits_summary}")

asyncio.run(main())

## Next Steps

- **[Agent-to-Agent Communication](a2a-communication.ipynb)** — agents that message each other directly
- **[Parallel Agent Execution](parallel-agent-execution.ipynb)** — run independent pipeline stages concurrently
- **[Crew Content Pipeline](crew-content-pipeline.ipynb)** — let the `Crew` manage task assignment automatically